# 模型性能测试

用于手动测试 API 调用大模型和本地模型的翻译性能。

In [12]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("Openrouter_API_KEY"),    
)

## API 模型测试

In [13]:
def test_model(model: str, messages: list, temperature: float = 0.3) -> dict:
    """测试单个模型，返回耗时和结果"""
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    elapsed = time.perf_counter() - start
    return {
        "model": model,
        "time": round(elapsed, 2),
        "content": resp.choices[0].message.content,
        "usage": {
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
        }
    }

In [14]:
# 测试翻译任务：长难句分析
messages = [
    {"role": "system", "content": (
        "你是一个专业英语翻译与分析助手。对用户输入的英文长难句，按以下格式输出：\n"
        "1. 【直译】逐词直译成中文，尽量保留原文语序和结构\n"
        "2. 【文法拆解】分析句子结构：主句、从句、修饰成分等\n"
        "3. 【词汇短语】列出实用或较难理解的单词和短语，附中文释义和简要说明\n"
    )},
    {"role": "user", "content": (
        "Had I not been so preoccupied with the minutiae of the project, "
        "which, as it turned out, were largely inconsequential to its ultimate success, "
        "I might have recognized sooner that the overarching strategy, "
        "brilliant though it was in conception, "
        "would have been rendered utterly futile "
        "had the market conditions shifted, as they eventually did, "
        "in a direction that no one had anticipated."
    )},
]

# 可替换为你想测试的模型
models = [
    #"openai/gpt-4o-mini",
    #"openai/gpt-4o",
     "anthropic/claude-sonnet-5",
     "deepseek/deepseek-v4-flash",
     "deepseek/deepseek-v4-pro",
]

for m in models:
    result = test_model(m, messages)
    print(f"模型: {result['model']}")
    print(f"耗时: {result['time']}s")
    print(f"Token: {result['usage']}")
    print(result['content'])
    print("-" * 50)

模型: anthropic/claude-sonnet-5
耗时: 24.94s
Token: {'prompt_tokens': 255, 'completion_tokens': 1727}
# 【直译】

如果我不是那样/全神贯注于/这个项目的/琐碎细节，/（这些细节）事实证明/在很大程度上/无关紧要/对于其/最终的成功，/我可能会/更早地/意识到/那个/总体战略，/尽管/在构思上/它是/精妙的，/本会被/弄得/彻底无用，/如果/市场条件/发生了变化，/正如/它们最终/确实（变化了）那样，/朝着一个/没有人/曾预料到的/方向。

# 【文法拆解】

**句子类型**：多重虚拟条件句（含两个倒装条件从句）+ 宾语从句嵌套

**主干结构**：
- **倒装条件句1**（表过去虚拟）：*Had I not been so preoccupied with the minutiae of the project* = If I had not been so preoccupied...
- **主句**：*I might have recognized sooner that...*
- **宾语从句**（recognized 的宾语）：*that the overarching strategy... would have been rendered utterly futile...*

**嵌套修饰成分**：
1. *which, as it turned out, were largely inconsequential to its ultimate success*
   → 非限制性定语从句，修饰 "the minutiae"；其中 "as it turned out" 是插入语

2. *brilliant though it was in conception*
   → 让步状语从句（倒装结构，though前置形容词表强调），修饰 strategy

3. *had the market conditions shifted...*
   → 倒装条件句2（= if the market conditions had shifted），是宾语从句中的条件状语

4. *as they eventually did*
   → 插入的比较状语从句

## 本地模型测试（Ollama）

本地通过 Ollama 运行模型，兼容 OpenAI 接口。当前已安装：`qwen2.5:3b`

In [15]:
# 初始化 Ollama 客户端（本地，无需 API Key）
local_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama 不校验 key，但需要填一个占位值
)

In [16]:
def test_local_model(model: str, messages: list, temperature: float = 0.3) -> dict:
    """测试本地模型，返回耗时和结果"""
    start = time.perf_counter()
    resp = local_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    elapsed = time.perf_counter() - start
    return {
        "model": model,
        "time": round(elapsed, 2),
        "content": resp.choices[0].message.content,
        "usage": {
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
        }
    }

In [17]:
# 测试本地模型翻译
local_models = [
    "qwen2.5:3b",
    # 如果有其他本地模型，在这里添加
]

for m in local_models:
    result = test_local_model(m, messages)
    print(f"模型: {result['model']}")
    print(f"耗时: {result['time']}s")
    print(f"译文: {result['content']}")
    print(f"Token: {result['usage']}")
    print("-" * 50)

模型: qwen2.5:3b
耗时: 14.17s
译文: 【直译】Had I not been so preoccupied with the minutiae of the project, which, as it turned out, were largely inconsequential to its ultimate success, I might have recognized sooner that the overarching strategy, brilliant though it was in conception, would have been rendered utterly futile had the market conditions shifted, as they eventually did, in a direction that no one had anticipated.
【文法拆解】这是一个复杂的复合句，包含多个从句。主句是“I might have recognized sooner that the overarching strategy...”，其中“had the market conditions shifted, as they eventually did”是一个条件状语从句，“in a direction that no one had anticipated”是一个结果状语从句。
【词汇短语】
1. minutiae（n）：细节，琐碎之事
2. preoccupied（v）：忙于，分心于
3. inconsequential（adj）：无关紧要的，不重要的
4. conceived（v）：构思，设想
5. futile（adj）：徒劳的，无用的
6. anticipated（v）：预料，预期
Token: {'prompt_tokens': 181, 'completion_tokens': 232}
--------------------------------------------------
